# RAG Pipeline with Hugging Face Docs, LangChain, FAISS, and FLAN-T5

This notebook builds a small Retrieval-Augmented Generation (RAG) pipeline over the `m-ric/huggingface_doc` dataset, following the task list:

1. Setup and imports
2. Load the dataset
3. Convert rows into LangChain `Document` objects
4. Chunk the documents
5. Build the vector store and retriever
6. Retrieval sanity check
7. RAG question answering

## 1. Setup and imports

Install dependencies, then import the dataset loader, Hugging Face pipeline utilities, and the LangChain pieces needed for chunking, embedding, retrieval, and QA.

In [ ]:
# Run this once. Restart the kernel afterward if any packages were newly installed.
!pip install -q langchain
!pip install -q torch
!pip install -q transformers
!pip install -q sentence-transformers
!pip install -q datasets
!pip install -q faiss-cpu
!pip install -U langchain-community


In [ ]:
from langchain.document_loaders import HuggingFaceDatasetLoader

from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA


## 2. Load the dataset

Load `databricks/databricks-dolly-15k` with `HuggingFaceDatasetLoader` and inspect one loaded document.

In [ ]:
!pip install -Uq datasets

dataset_name = "databricks/databricks-dolly-15k"
page_content_column = "context"
loader = HuggingFaceDatasetLoader(dataset_name, page_content_column)
data = loader.load()
print("Loaded documents:", len(data))
print(data[:2])


In [ ]:
# Inspect the first loaded document
print(data[0])


## 3. Split the loaded documents into chunks

Each `Document` chunk stores its text in `page_content`. Overlapping chunking preserves context while keeping each piece small enough for embedding and retrieval.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
# Optional: use a smaller subset for faster experimentation
subset = data[:200]
docs = text_splitter.split_documents(subset)
print(f"Total chunks: {len(docs)}")
print("Example chunk content (first 300 chars):\n", docs[0].page_content[:300])


> **Note:** The full dataset has several thousand rows. For faster experimentation in this notebook, the chunking and indexing steps below use a subset (`docs[:200]`). Increase or remove the slice once you're happy with the pipeline and have time/compute to spare.

## 4. Review the chunked documents

The loaded documents have been split into overlapping chunks with `RecursiveCharacterTextSplitter`. Review a sample chunk to verify the split quality before indexing.

In [ ]:
print(f"Using {len(docs)} chunks for indexing.")
print("--- Sample chunk ---")
print(docs[0].page_content[:300])


In [ ]:
# The chunks are already split and ready for embedding
print("First chunk content preview:")
print(docs[0].page_content[:300])


**Observation prompt:** Verify that chunk boundaries preserve coherent context and that chunks are not too short. Good chunks should contain complete sentences or small paragraphs, and overlap enough to keep retrieval robust.

## 5. Build the vector store and retriever

Embed the chunks with `sentence-transformers/all-MiniLM-L6-v2`, build a FAISS index, and create retrievers with `k=4` (the default to use going forward), plus `k=2` and `k=6` for comparison.

In [ ]:
modelPath = "sentence-transformers/all-MiniLM-L6-v2"
model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": False}
embeddings = HuggingFaceEmbeddings(
    model_name=modelPath,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)
db = FAISS.from_documents(docs, embeddings)
retriever_k2 = db.as_retriever(search_kwargs={"k": 2})
retriever_k4 = db.as_retriever(search_kwargs={"k": 4})
retriever_k6 = db.as_retriever(search_kwargs={"k": 6})

print("Vector store built with", db.index.ntotal, "vectors.")


## 6. Retrieval sanity check (required)

Before generating any answers, manually inspect retrieved chunks for a test question across the three `k` values. If chunks look irrelevant, go back and adjust chunking parameters and/or `k`.

In [ ]:
test_question = "How do I load a dataset with the datasets library?"

for label, retriever in [("k=2", retriever_k2), ("k=4", retriever_k4), ("k=6", retriever_k6)]:
    print(f"\n=== Retrieval with {label} ===")
    results = retriever.get_relevant_documents(test_question)
    for i, r in enumerate(results):
        print(f"\n--- Chunk {i + 1} ---")
        print(r.page_content[:300])


**Checklist before moving on:**
- Do the retrieved chunks actually mention loading datasets / the `datasets` library?
- Does increasing `k` from 2 → 6 add genuinely useful chunks, or mostly noise?
- Do the `source` values look meaningful (not all `unknown`)?

If the answer to any of these is "no," go back to step 4 and try a different `chunk_size`/`chunk_overlap`, or rebuild the index in step 5 with a larger `doc_subset`, before continuing.

## 7. RAG question answering

Set up a small local LLM (`google/flan-t5-small`), wrap it in a `RetrievalQA` chain with the `k=4` retriever, ask several questions, and print both the answer and the sources of the retrieved chunks.

In [ ]:
hf_pipe = pipeline(
    "text2text-generation",
    model="google/flan-t5-small",
    max_new_tokens=256,
)
llm = HuggingFacePipeline(pipeline=hf_pipe)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever_k4,
    return_source_documents=True,
)


In [ ]:
questions = [
    "How do I load a dataset with the datasets library?",
    "What is a Hugging Face pipeline?",
    "How do I push a model to the Hugging Face Hub?",
]

for q in questions:
    result = qa_chain({"query": q})
    print(f"\nQ: {q}")
    print(f"A: {result['result']}")
    print("Sources used:")
    for doc in result["source_documents"]:
        print(f"  - {doc.page_content[:200]}")
    print("-" * 80)


**Evaluating the answers:** `flan-t5-small` is a small, weak model, so answers may be terse or occasionally miss the point even with good context. Focus less on eloquence and more on whether:
- The printed **sources** are actually relevant to the question (this tells you retrieval is working).
- The **answer** at least paraphrases something present in the retrieved chunks, rather than sounding like unrelated generic text (this tells you the model is grounding on the dataset rather than guessing).